In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/SAMPLE_DATA.csv
/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/TRAIN_DATA.csv
/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/TEST_DATA.csv


In [2]:
# 1. Import Libraries

import os
import glob
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

In [3]:
# 2. Find Dataset Files

TRAIN_PATH = "/kaggle/input/egspec-cyberguard-2026-ai-ds/TRAIN_DATA.csv"
TEST_PATH = "/kaggle/input/egspec-cyberguard-2026-ai-ds/TEST_DATA.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"

TARGET = "Attack_Label"
ID_COLUMN = "Flow_ID"

In [4]:
# 3. Set Dataset and Output Paths

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/SAMPLE_DATA.csv
/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/TRAIN_DATA.csv
/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/TEST_DATA.csv


In [5]:
# 4. Load Train and Test Data

train_files = glob.glob(
    "/kaggle/input/**/TRAIN_DATA.csv",
    recursive=True
)

test_files = glob.glob(
    "/kaggle/input/**/TEST_DATA.csv",
    recursive=True
)

print(train_files)
print(test_files)

['/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/TRAIN_DATA.csv']
['/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/TEST_DATA.csv']


In [6]:
# 5. Explore Dataset

TRAIN_PATH = train_files[0]
TEST_PATH = test_files[0]

OUTPUT_PATH = "/kaggle/working/submission.csv"

TARGET = "Attack_Label"
ID_COLUMN = "Flow_ID"

print(TRAIN_PATH)
print(TEST_PATH)

/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/TRAIN_DATA.csv
/kaggle/input/competitions/egspec-cyberguard-2026-ai-ds/TEST_DATA.csv


In [7]:
# 6. Clean Missing and Infinite Values

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(train_df.shape)
print(test_df.shape)

display(train_df.head())
display(test_df.head())

(100000, 50)
(25000, 49)


,Flow_ID,Flow_Duration,Protocol,Source_Port,Destination_Port,Total_Fwd_Packets,Total_Bwd_Packets,Total_Fwd_Bytes,Total_Bwd_Bytes,Fwd_Packet_Length_Mean,...,Init_Win_Bytes_Fwd,Init_Win_Bytes_Bwd,Min_Packet_Length,Max_Packet_Length,Avg_Packet_Size,Flow_IAT_Mean,Flow_IAT_Std,Fwd_IAT_Mean,Bwd_IAT_Mean,Attack_Label
0,1,21387262,TCP,1047,443,307,2177,620518,1090701,1432.46,...,48964,2740,35,473,840.01,19520.56,4051.54,12891.61,80518.77,Benign
1,2,713492,TCP,6274,21,2336,1946,1678689,1302132,107.61,...,26405,50070,19,1195,1117.11,76302.75,4487.56,68760.85,65613.03,Malicious
2,3,26215687,TCP,21247,25,4734,2062,543350,1160622,50.98,...,17633,2980,24,913,1026.58,66001.32,23219.10,14111.03,5275.71,Malicious
3,4,88215921,TCP,42102,3389,1572,154,867569,1987193,714.62,...,24975,8279,1,853,167.21,5473.61,34968.88,40463.93,71281.73,Benign
4,5,84194330,TCP,9938,443,1376,4680,580368,595673,1208.34,...,3153,21416,17,451,1028.52,98595.21,352.44,83365.17,97040.36,Malicious


,Flow_ID,Flow_Duration,Protocol,Source_Port,Destination_Port,Total_Fwd_Packets,Total_Bwd_Packets,Total_Fwd_Bytes,Total_Bwd_Bytes,Fwd_Packet_Length_Mean,...,Down_Up_Ratio,Init_Win_Bytes_Fwd,Init_Win_Bytes_Bwd,Min_Packet_Length,Max_Packet_Length,Avg_Packet_Size,Flow_IAT_Mean,Flow_IAT_Std,Fwd_IAT_Mean,Bwd_IAT_Mean
0,100001,91401494,TCP,40543,25,4983,3660,1054722,1541841,1049.82,...,5.86,51764,46362,24,734,117.26,58138.81,937.91,25554.68,76699.44
1,100002,115971339,TCP,64728,8080,4637,1095,658528,1476336,234.31,...,8.42,28591,55465,33,911,542.35,34729.76,2907.33,2080.82,94973.04
2,100003,8237653,TCP,19913,443,3729,3803,1105283,1345420,1225.30,...,4.19,2304,35742,33,221,98.14,93206.18,17475.16,41892.01,5122.31
3,100004,25452646,TCP,58130,21,209,4620,1180336,9532,1189.62,...,8.54,45851,51822,32,148,593.03,60526.02,23886.53,91909.60,40167.97
4,100005,26010270,UDP,7362,80,2788,1842,754821,1385046,903.20,...,6.95,38883,17133,3,530,1446.06,71228.43,29880.53,89308.22,78118.30


In [8]:
# 7. Handle Categorical Data

print(train_df.columns.tolist())
print(test_df.columns.tolist())

['Flow_ID', 'Flow_Duration', 'Protocol', 'Source_Port', 'Destination_Port', 'Total_Fwd_Packets', 'Total_Bwd_Packets', 'Total_Fwd_Bytes', 'Total_Bwd_Bytes', 'Fwd_Packet_Length_Mean', 'Bwd_Packet_Length_Mean', 'Packet_Length_Mean', 'Packet_Length_Std', 'Packet_Length_Max', 'Packet_Length_Min', 'Flow_Bytes_Per_Sec', 'Flow_Packets_Per_Sec', 'Fwd_Packets_Per_Sec', 'Bwd_Packets_Per_Sec', 'Fwd_Header_Length', 'Bwd_Header_Length', 'SYN_Flag_Count', 'ACK_Flag_Count', 'RST_Flag_Count', 'PSH_Flag_Count', 'FIN_Flag_Count', 'URG_Flag_Count', 'Active_Mean', 'Active_Max', 'Active_Min', 'Idle_Mean', 'Idle_Max', 'Idle_Min', 'Subflow_Fwd_Packets', 'Subflow_Bwd_Packets', 'Subflow_Fwd_Bytes', 'Subflow_Bwd_Bytes', 'Avg_Fwd_Segment_Size', 'Avg_Bwd_Segment_Size', 'Down_Up_Ratio', 'Init_Win_Bytes_Fwd', 'Init_Win_Bytes_Bwd', 'Min_Packet_Length', 'Max_Packet_Length', 'Avg_Packet_Size', 'Flow_IAT_Mean', 'Flow_IAT_Std', 'Fwd_IAT_Mean', 'Bwd_IAT_Mean', 'Attack_Label']
['Flow_ID', 'Flow_Duration', 'Protocol', 'Sour

In [9]:
# 8. Prepare Features and Target

print(train_df[TARGET].value_counts())

print(
    train_df[TARGET]
    .value_counts(normalize=True)
)

Attack_Label
Benign       70147
Malicious    29853
Name: count, dtype: int64
Attack_Label
Benign       0.70147
Malicious    0.29853
Name: proportion, dtype: float64


In [10]:
# 9. Split Training and Validation Data

if TARGET in test_df.columns:
    test_df = test_df.drop(
        columns=[TARGET]
    )

train_df = train_df.replace(
    [np.inf, -np.inf],
    np.nan
)

test_df = test_df.replace(
    [np.inf, -np.inf],
    np.nan
)

In [11]:
# 10. Build CatBoost Model

numeric_columns = (
    train_df
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

for column in numeric_columns:

    if column == ID_COLUMN:
        continue

    if column in test_df.columns:

        median_value = train_df[column].median()

        train_df[column] = (
            train_df[column]
            .fillna(median_value)
        )

        test_df[column] = (
            test_df[column]
            .fillna(median_value)
        )

In [12]:
# 11. Train Model

categorical_columns = (
    train_df
    .drop(
        columns=[TARGET],
        errors="ignore"
    )
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

for column in categorical_columns:

    train_df[column] = (
        train_df[column]
        .fillna("Unknown")
        .astype(str)
    )

    if column in test_df.columns:

        test_df[column] = (
            test_df[column]
            .fillna("Unknown")
            .astype(str)
        )

In [13]:
# 12. Validate Model Performance

print(
    train_df.isnull()
    .sum()
    .sum()
)

print(
    test_df.isnull()
    .sum()
    .sum()
)

0
0


In [14]:
# 13. Check Feature Importance

test_ids = test_df[
    ID_COLUMN
].copy()

FEATURES = [
    column
    for column in test_df.columns
    if column != ID_COLUMN
]

X = train_df[
    FEATURES
].copy()

y = train_df[
    TARGET
].copy()

X_test = test_df[
    FEATURES
].copy()

print(X.shape)
print(y.shape)
print(X_test.shape)

(100000, 48)
(100000,)
(25000, 48)


In [15]:
# 14. Train Final Model on Full Dataset

categorical_features = (
    X
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

print(categorical_features)

['Protocol']


In [16]:
# 15. Predict Test Data

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)
print(y_train.shape)
print(y_valid.shape)

(80000, 48)
(20000, 48)
(80000,)
(20000,)


In [17]:
# 16. Create Submission File

model = CatBoostClassifier(
    iterations=1200,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="F1",
    l2_leaf_reg=5,
    random_seed=42,
    random_strength=0.5,
    thread_count=-1,
    use_best_model=True,
    allow_writing_files=False,
    verbose=100
)

In [18]:
# 17. Validate Submission

model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(
        X_valid,
        y_valid
    ),
    early_stopping_rounds=150
)

0:	learn: 0.9904706	test: 0.9907885	best: 0.9907885 (0)	total: 147ms	remaining: 2m 56s
100:	learn: 0.9905543	test: 0.9912999	best: 0.9912999 (1)	total: 6.88s	remaining: 1m 14s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9912999409
bestIteration = 1

Shrink model to first 2 iterations.


CatBoostClassifier(allow_writing_files=False, depth=8, eval_metric='F1', iterations=1200, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=42, random_strength=0.5, use_best_model=True, verbose=100)

In [19]:
valid_predictions = model.predict(
    X_valid
)

valid_predictions = np.asarray(
    valid_predictions
).reshape(-1)

In [20]:
accuracy = accuracy_score(
    y_valid,
    valid_predictions
)

f1 = f1_score(
    y_valid,
    valid_predictions,
    pos_label="Malicious"
)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

Accuracy: 0.99485
F1 Score: 0.9912999408733846


In [21]:
print(
    classification_report(
        y_valid,
        valid_predictions,
        digits=6
    )
)

              precision    recall  f1-score   support

      Benign   0.992712  1.000000  0.996342     14029
   Malicious   1.000000  0.982750  0.991300      5971

    accuracy                       0.994850     20000
   macro avg   0.996356  0.991375  0.993821     20000
weighted avg   0.994888  0.994850  0.994837     20000



In [22]:
print(
    confusion_matrix(
        y_valid,
        valid_predictions
    )
)

[[14029     0]
 [  103  5868]]


In [23]:
feature_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model.get_feature_importance()
})

feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    feature_importance.head(20)
)

,Feature,Importance
0,Flow_Packets_Per_Sec,57.714077
1,SYN_Flag_Count,36.229178
2,Packet_Length_Mean,0.769149
3,Bwd_Header_Length,0.748612
4,Flow_Duration,0.726504
5,Subflow_Fwd_Bytes,0.723800
6,Init_Win_Bytes_Bwd,0.661698
7,Total_Bwd_Bytes,0.611830
8,Fwd_Packets_Per_Sec,0.546509
9,Bwd_IAT_Mean,0.413283


In [24]:
best_iteration = model.get_best_iteration()

if best_iteration is None or best_iteration < 1:
    best_iteration = 600
else:
    best_iteration += 1

print(best_iteration)

2


In [25]:
final_model = CatBoostClassifier(
    iterations=best_iteration,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    l2_leaf_reg=5,
    random_seed=42,
    random_strength=0.5,
    thread_count=-1,
    allow_writing_files=False,
    verbose=100
)

In [26]:
final_model.fit(
    X,
    y,
    cat_features=categorical_features
)

0:	learn: 0.5530758	total: 68ms	remaining: 68ms
1:	learn: 0.4521763	total: 130ms	remaining: 0us


CatBoostClassifier(allow_writing_files=False, depth=8, iterations=2, l2_leaf_reg=5, learning_rate=0.05, loss_function='Logloss', random_seed=42, random_strength=0.5, verbose=100)

In [27]:
test_predictions = final_model.predict(
    X_test
)

test_predictions = np.asarray(
    test_predictions
).reshape(-1)

In [28]:
print(
    pd.Series(
        test_predictions
    ).value_counts()
)

Benign    25000
Name: count, dtype: int64


In [29]:
submission = pd.DataFrame({
    ID_COLUMN: test_ids,
    TARGET: test_predictions
})

display(
    submission.head(20)
)

print(
    submission.shape
)

,Flow_ID,Attack_Label
0,100001,Benign
1,100002,Benign
2,100003,Benign
3,100004,Benign
4,100005,Benign
5,100006,Benign
6,100007,Benign
7,100008,Benign
8,100009,Benign
9,100010,Benign


(25000, 2)


In [30]:
assert len(submission) == len(test_df)

assert (
    submission[ID_COLUMN]
    .isnull()
    .sum()
    == 0
)

assert (
    submission[TARGET]
    .isnull()
    .sum()
    == 0
)

In [31]:
# 18. Save Final Submission

submission.to_csv(
    OUTPUT_PATH,
    index=False
)

print(OUTPUT_PATH)

/kaggle/working/submission.csv


In [32]:
final_submission = pd.read_csv(
    "/kaggle/working/submission.csv"
)

display(
    final_submission.head(20)
)

print(
    final_submission[
        TARGET
    ].value_counts()
)

,Flow_ID,Attack_Label
0,100001,Benign
1,100002,Benign
2,100003,Benign
3,100004,Benign
4,100005,Benign
5,100006,Benign
6,100007,Benign
7,100008,Benign
8,100009,Benign
9,100010,Benign


Attack_Label
Benign    25000
Name: count, dtype: int64
